# 01 · 기본 데이터 확인

DACON 236736 로봇 영상 생성 데이터의 구성과 샘플을 살펴봅니다.
위에서부터 차례대로 실행하세요. 원본 파일을 수정하지 않으며 모델을 학습하지 않습니다.

평가 데이터는 형식 확인에만 사용합니다. 학습·정규화·검증 데이터로 사용하지 않습니다.
실행 출력에는 원본 샘플이 포함되므로 GitHub에 올리기 전 모든 출력을 지우세요.


In [ ]:
from pathlib import Path
import json
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Video

# 리포 루트 또는 notebooks/에서 실행할 수 있습니다.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists() and (p / "scripts/inspect_data.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("이 리포지토리에서 노트북을 실행하세요.")
RAW = PROJECT_ROOT / "data/raw"  # 다른 경로를 사용한다면 수정하세요.
TRAIN = RAW / "data/train"
EVAL = RAW / "data/eval"
if not TRAIN.is_dir() or not EVAL.is_dir():
    raise FileNotFoundError(f"공식 open.zip을 {RAW} 아래에 압축 해제하세요.")
print("프로젝트:", PROJECT_ROOT)
print("데이터:", RAW)


## 1. 전체 파일 구성

파일 종류별 개수와 디스크 크기를 확인합니다. 수치는 현재 로컬 파일 기준입니다.


In [ ]:
files = sorted(p for p in RAW.rglob("*") if p.is_file())
inventory = pd.DataFrame([{"path": str(p.relative_to(RAW)), "section": p.relative_to(RAW).parts[0], "extension": p.suffix or "(none)", "bytes": p.stat().st_size} for p in files])
display(inventory.groupby(["section", "extension"]).agg(files=("path", "count"), size_MB=("bytes", lambda s: s.sum() / 1e6)).round(2))
print(f"총 {len(files):,}개 파일 / {inventory.bytes.sum() / 1e9:.2f} GB")


## 2. 학습 데이터 메타데이터

하나의 데이터셋은 여러 조작 에피소드로 구성됩니다. 프레임 수·FPS·해상도는 메타데이터 기준입니다.


In [ ]:
info_paths = sorted(TRAIN.rglob("meta/info.json"))
rows = []
for path in info_paths:
    info = json.loads(path.read_text())
    videos = {k: v for k, v in info["features"].items() if v.get("dtype") == "video"}
    rows.append({"dataset": str(path.parent.parent.relative_to(TRAIN)), "robot": info.get("robot_type"), "episodes": info.get("total_episodes"), "frames": info.get("total_frames"), "fps": info.get("fps"), "action_shape": str(info["features"]["action"]["shape"]), "video_keys": ", ".join(videos), "video_shapes": str([v["shape"] for v in videos.values()])})
datasets = pd.DataFrame(rows)
display(datasets.head(10))
display(datasets.groupby("fps").agg(datasets=("dataset", "count"), episodes=("episodes", "sum")))
print(f"{len(datasets)}묶음 / {datasets.episodes.sum():,}에피소드 / {datasets.frames.sum():,}프레임")


In [ ]:
lengths = []
for path in sorted(TRAIN.rglob("meta/episodes.jsonl")):
    lengths.extend(json.loads(line)["length"] for line in path.read_text().splitlines() if line.strip())
display(pd.Series(lengths, name="episode_frames").describe(percentiles=[0.5, 0.95]))
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths, bins=50)
ax.set(title="Training episode length (metadata)", xlabel="Length (frames)", ylabel="Episodes (count)")
plt.show()
print("16프레임 미만:", sum(n < 16 for n in lengths))


## 3. 평가 이미지와 행동 배열

sample ID로 이미지와 행동을 연결합니다. 행동 `(16, 6)`은 16시점의 6차원 조건이며, 아래 그래프의 값은 원본 수치입니다. 물리 단위는 여기서 가정하지 않습니다.


In [ ]:
image_paths = sorted((EVAL / "images").glob("*.png"))
action_paths = sorted((EVAL / "actions").glob("*.npy"))
image_ids = {p.stem for p in image_paths}
action_ids = {p.stem for p in action_paths}
assert image_ids == action_ids, "이미지/행동 sample ID가 다릅니다."
print("평가 입력 쌍:", len(image_ids))
print("행동 형식:", Counter((np.load(p, allow_pickle=False).shape, str(np.load(p, allow_pickle=False).dtype)) for p in action_paths))
assert all(np.isfinite(np.load(p, allow_pickle=False)).all() for p in action_paths), "NaN/Inf를 발견했습니다."
sample_id = image_paths[0].stem  # 다른 sample ID로 바꿔 볼 수 있습니다.
with Image.open(EVAL / "images" / f"{sample_id}.png") as image:
    display(image.copy())
actions = np.load(EVAL / "actions" / f"{sample_id}.npy", allow_pickle=False)
print(sample_id, actions.shape, actions.dtype)
display(pd.DataFrame(actions, columns=[f"action_{i}" for i in range(actions.shape[1])]))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for dim in range(actions.shape[1]):
    ax.plot(np.arange(len(actions)), actions[:, dim], marker=".", label=f"action_{dim}")
ax.set(title=f"Evaluation action sequence: {sample_id}", xlabel="Sequence index (step)", ylabel="Action value (raw units)")
ax.legend(ncol=3)
plt.show()


## 4. 학습 에피소드 하나 읽기

Parquet의 action, observation.state와 시간 인덱스를 확인합니다. 전체 학습 데이터를 메모리에 올리지 않습니다.


In [ ]:
info_path = info_paths[0]  # 다른 데이터셋을 보고 싶으면 인덱스를 변경하세요.
dataset_root = info_path.parent.parent
info = json.loads(info_path.read_text())
parquet_path = sorted(dataset_root.rglob("*.parquet"))[0]
episode = pq.read_table(parquet_path).to_pandas()
print("데이터셋:", dataset_root.relative_to(TRAIN))
print("Parquet:", parquet_path.name)
print("컬럼:", episode.columns.tolist())
display(episode.head())
print("행동 항목:", info["features"]["action"].get("names"))
print("FPS:", info["fps"])
train_actions = np.stack(episode["action"].to_numpy())
states = np.stack(episode["observation.state"].to_numpy())
print("action / state:", train_actions.shape, states.shape)
print("유한값:", np.isfinite(train_actions).all(), np.isfinite(states).all())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
names = info["features"]["action"].get("names") or [f"joint_{i}" for i in range(train_actions.shape[1])]
if not isinstance(names, list):
    names = [f"joint_{i}" for i in range(train_actions.shape[1])]
for dim in range(train_actions.shape[1]):
    label = names[dim] if dim < len(names) else f"joint_{dim}"
    axes[0].plot(episode["timestamp"], train_actions[:, dim], label=label)
    axes[1].plot(episode["timestamp"], states[:, dim], label=label)
axes[0].set(title=f"Training action: {parquet_path.stem}", ylabel="Action (raw units)")
axes[1].set(title="Observed robot state", xlabel="Timestamp (seconds)", ylabel="State (raw units)")
axes[0].legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()


## 5. 대응 영상 확인 (선택)

아래 셀은 로컬 MP4를 연결해 재생합니다. 브라우저가 AV1을 지원하지 않으면 재생되지 않을 수 있습니다. 영상 전체의 디코딩·시간 정렬 검증을 대신하지는 않습니다.


In [ ]:
video_paths = sorted(dataset_root.glob(f"videos/**/{parquet_path.stem}.mp4"))
print("대응 영상:", [str(p.relative_to(RAW)) for p in video_paths])
for path in video_paths:
    display(Video(filename=str(path), embed=False, width=640))


## 다음 확인 사항

- 같은 에피소드의 프레임을 학습/검증에 섞지 않기
- FPS·해상도·코덱 차이를 처리하고 영상 프레임과 timestamp를 맞추기
- 짧은 에피소드의 제외/패딩 정책을 정하기
- 정규화 통계는 학습 분할에서만 계산하기

이 노트북은 탐색용이며, 모델 학습이나 제출 도구 실행은 포함하지 않습니다.
